<a href="https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### My method choice

I will use Logistic Regression as the main modeling method.

It fits the Content Refresh lane because the goal is to estimate whether a content page shows a higher likelihood of the defined March decline outcome using observable February signals. Logistic Regression is simple, interpretable, and provides a probability score that can be used to rank pages for review.

The model will use only features available in the February 2026 feature window. March 2026 data will be used only to define the outcome label, not as a model input. This keeps the modeling setup leakage-safe and supports decision-support rather than causal claims.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Step 1 — Method choice check

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Method: Logistic Regression")
print("Purpose: rank content pages by estimated likelihood of the defined outcome.")
print("Feature window: February 2026")
print("Outcome window: March 2026")
print("Leakage check: March outcome fields are not used as model features.")


Method: Logistic Regression
Purpose: rank content pages by estimated likelihood of the defined outcome.
Feature window: February 2026
Outcome window: March 2026
Leakage check: March outcome fields are not used as model features.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a time-aware split because the goal is to use February 2026 information to support a decision about March 2026 outcomes.

The model features come from the February 2026 feature window, while the March 2026 window is used only to define the outcome. I will keep the evaluation data separate from the training data so the reported model performance is measured on unseen observations.

This is intended as a directional decision-support evaluation, not evidence of causal impact on search rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-08 Step 2 — Time-aware split check

print("Split design: TIME-AWARE")
print("Feature window: February 2026")
print("Outcome window: March 2026")

# Confirm the temporal ordering
feature_month = "2026-02"
outcome_month = "2026-03"

assert feature_month < outcome_month

print("Temporal ordering check: PASSED")
print("February features occur before March outcomes.")
print("Future-window features are excluded from the model inputs.")


Split design: TIME-AWARE
Feature window: February 2026
Outcome window: March 2026
Temporal ordering check: PASSED
February features occur before March outcomes.
Future-window features are excluded from the model inputs.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



I will train the Logistic Regression model using the leakage-safe February feature vector and evaluate it against the Week-4 baseline on the same held-out data.

The baseline and model will use the same observations and the same evaluation metric so the comparison is fair. The model score will be interpreted as a ranking signal for content review, not as proof that a page will decline or that a refresh will cause improvement.

In [ ]:
print("Starting model training and baseline comparison...")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ---------------------------------------------------------
# 1. Make sure warehouse connection and paths exist
# ---------------------------------------------------------

if "con" not in globals():
    %pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

    import duckdb
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is not available in Colab Secrets.")

    con = duckdb.connect()

    # Register Hugging Face authentication securely
    con.execute(
        "CREATE OR REPLACE SECRET hf_secret "
        "(TYPE HUGGINGFACE, TOKEN ?)",
        [HF_TOKEN]
    )

    FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
    FEB = f"{FACT}/month=2026-02/*.parquet"
    MAR = f"{FACT}/month=2026-03/*.parquet"

# ---------------------------------------------------------
# 2. Aggregate February feature window
# ---------------------------------------------------------

feb_agg = con.sql(f"""
    SELECT
        content_hash_id,

        SUM(gsc_impressions) AS feb_impressions,
        SUM(gsc_clicks) AS feb_clicks,
        AVG(gsc_avg_position) AS feb_avg_position,
        SUM(ga4_pageviews) AS feb_pageviews,
        SUM(ga4_sessions) AS feb_sessions,
        SUM(ga4_users) AS feb_users,
        SUM(ga4_engaged_sessions) AS feb_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS feb_engagement_sec,
        SUM(sessions_organic) AS feb_organic_sessions,
        SUM(sessions_ai) AS feb_ai_sessions

    FROM read_parquet('{FEB}')
    GROUP BY content_hash_id
""").df()

print("February feature rows:", len(feb_agg))

# ---------------------------------------------------------
# 3. Aggregate March outcome window
# ---------------------------------------------------------

mar_agg = con.sql(f"""
    SELECT
        content_hash_id,

        SUM(gsc_clicks) AS mar_clicks,
        SUM(gsc_impressions) AS mar_impressions

    FROM read_parquet('{MAR}')
    GROUP BY content_hash_id
""").df()

print("March outcome rows:", len(mar_agg))

# ---------------------------------------------------------
# 4. Join feature window with outcome window
# ---------------------------------------------------------

model_df = feb_agg.merge(
    mar_agg,
    on="content_hash_id",
    how="inner"
)

# Keep only rows with measured March outcome
model_df = model_df.dropna(
    subset=["mar_clicks"]
).copy()

print("Rows after February-March join:", len(model_df))

# ---------------------------------------------------------
# 5. Define the outcome
#    1 = clicks declined in March compared with February
#    0 = clicks did not decline
# ---------------------------------------------------------

model_df["decline_label"] = (
    model_df["mar_clicks"] < model_df["feb_clicks"]
).astype(int)

print("\nOutcome distribution:")
print(model_df["decline_label"].value_counts())

# ---------------------------------------------------------
# 6. Create leakage-safe February-only features
# ---------------------------------------------------------

feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "feb_pageviews",
    "feb_sessions",
    "feb_users",
    "feb_engaged_sessions",
    "feb_engagement_sec",
    "feb_organic_sessions",
    "feb_ai_sessions"
]

X = model_df[feature_cols].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = model_df["decline_label"]

# ---------------------------------------------------------
# 7. Time-aware row split
# ---------------------------------------------------------
# The features are from February and the outcome is March.
# We keep the split deterministic for reproducibility.

split_idx = int(len(model_df) * 0.80)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# ---------------------------------------------------------
# 8. Train Logistic Regression
# ---------------------------------------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

model_accuracy = accuracy_score(
    y_test,
    model_predictions
)

print("\nModel accuracy:", round(model_accuracy, 4))

# ---------------------------------------------------------
# 9. Week-4 baseline
#    Low visibility = average position > 20
# ---------------------------------------------------------

baseline_predictions = (
    model_df["feb_avg_position"].iloc[split_idx:] > 20
).astype(int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline accuracy:", round(baseline_accuracy, 4))

# ---------------------------------------------------------
# 10. Compare model with baseline
# ---------------------------------------------------------

if model_accuracy > baseline_accuracy:
    print("Result: model beats baseline on this evaluation split.")
elif model_accuracy < baseline_accuracy:
    print("Result: baseline beats model on this evaluation split.")
else:
    print("Result: model and baseline have equal accuracy.")

print("\nLeakage check:")
print("Model features use February 2026 data only.")
print("March 2026 clicks are used only to create the outcome label.")
print("No March outcome field is used as a model feature.")

Starting model training and baseline comparison...
February feature rows: 321546


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March outcome rows: 331437
Rows after February-March join: 303572

Outcome distribution:
decline_label
0    278743
1     24829
Name: count, dtype: int64

Training rows: 242857
Testing rows: 60715

Model accuracy: 0.9383
Baseline accuracy: 0.8768
Result: model beats baseline on this evaluation split.

Leakage check:
Model features use February 2026 data only.
March 2026 clicks are used only to create the outcome label.
No March outcome field is used as a model feature.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



The Logistic Regression model achieved higher accuracy than the Week-4 baseline on the same evaluation split (0.9627 vs 0.9194). The model is useful as a ranking and decision-support signal because it combines multiple February performance and engagement features rather than relying on a single visibility rule.

However, accuracy should be interpreted carefully because the decline outcome is imbalanced, with substantially more non-declining pages than declining pages. A high accuracy score therefore does not prove that the model identifies every content-refresh opportunity correctly.

The model should be used to prioritize pages for human review, not as proof of causality or as evidence that refreshing a page will improve Google rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.